## Procesamiento del Resultado de la API

En esta sección se normaliza la información obtenida desde la API para transformarla en un formato consistente y dejarla estructurada de la misma manera que el *dataframe* principal.


In [1]:
from __future__ import annotations

import json
import log_util
import pandas as pd
from pathlib import Path
from typing import Any, Dict, List, Optional
from __init__ import version, log_to_directory

logger = log_util.get_logger(__name__)
log_to_directory("logs")

PATH = './scopusv2.json'

In [2]:
fields = {
    'authors_info': 'Authors',
    'authors_info_with_id': 'Author full names',
    'authors_id': 'Author(s) ID',
    'dc:title': 'Title',
    'prism:coverDate': 'Date',
    'prism:volume': 'Volume',
    'prism:issueIdentifier': 'Issue',
    'citedby-count': 'Cited by',
    'prism:doi': 'DOI',
    'subtypeDescription': 'Document Type'
}

## __Necesito__

| Clave                  | Descripción (general) |
|-------------------------|-----------------------|
| `dc:title`            | Título del artículo/documento |
| `prism:publicationName`| Nombre de la revista o fuente |
| `prism:volume`        | Volumen de la publicación |
| `prism:issueIdentifier`| Número de la edición (issue) |
| `prism:pageRange`     | Páginas del artículo |
| `prism:coverDate`     | Fecha de publicación (ISO: YYYY-MM-DD) |
| `prism:doi`           | DOI del artículo |
| `citedby-count`       | Número de citas |
| `subtypeDescription`   | Descripción del subtipo (ej. Article, Review) |
| `author`               | Lista de autores |


In [6]:

class ElsevierSchemaError(RuntimeError):
    """It notes that the JSON response does not meet the contract expected from Elsevier/Scopus."""


class Elsjson: 
    """
    Parser/normalizer for Elsevier/Scopus Search Results JSON responses.

    Assumed contract (stable API):
    - The root must contain the key 'search-results' (dict).
    - 'search-results' must contain 'entry' (list of dicts). May be an empty list.
    - Each 'entry' contains metadata of the result (title, authors, etc.).

    Typical usage:
    parser = ElsevierSearchJSONParser(file_path="response.json", fields_map=FIELDS)
    df = parser.run()

    Parameters
    ----------
    file_path : str | Path | None
        Path to the local JSON file (when processing from disk).
    encoding : str
        Encoding of the JSON file (default 'utf-8').
    """

    def __init__(
        self, 
        path: Optional[str | Path] = None, 
        encoding: str = 'utf-8'
    ) -> None:
        self.path: Optional[Path] = Path(path) if path else None
        self._data: Optional[Dict[str, Any]] = None
        self.encoding: str = encoding


    @property
    def path(self):
        """ Gets the path"""
        if not self._path:
            raise AttributeError("path has not been loaded yet. Call self.path = Path(...)")
        return self._path
    
    @path.setter
    def path(self, path):
        """ Sets the path"""
        self._path = Path(path)

    @property
    def data(self):
        """Returns the parsed JSON data as a Python object (dict or list)."""
        if not self._data:
            raise AttributeError(
                "JSON data has not been loaded yet. Call load() or from_dict() first."
            )
        return self._data

    @data.setter
    def data(self, data):
        self._data = data
        self._validate_data()

    def load(self) -> None:
        """
        Reads the JSON file from the local path (used only when the file 
        is stored locally instead of in a database).
        """
        path = self.path
        logger.info("Reading Elsevier's JSON from %s", path)

        with path.open('r', encoding = 'utf-8') as f:
            self.data = json.load(f)

    def from_dict(self, payload: Dict[str, Any]) -> None:
        """
        Directly injects a JSON dict (e.g. request response).
        """
        self.data = payload

    def _validate_data(self) -> None:
        """
        Lightweight validation based on the API contract:
        - 'search-results' must exist and be a dict.
        - 'entry' must exist (list, possibly empty) or be coercible to one.
        """

        _ = len(self.data)

        if _ is None:
            logger.warning(
                "Elsevier response with no results: 0 data."
            )

    @staticmethod
    def preprocess_author(authors):
        return ';'.join(f'{auth["authname"]}' for auth in authors)

    @staticmethod
    def preprocess_author_with_id(authors):
        return ';'.join(f'{auth["authname"]} ({auth["authid"]})' for auth in authors)
    
    @staticmethod
    def preprocess_authorid(authors):
        return ';'.join(f'{auth["authid"]}' for auth in authors)

    def _enrich_entries(self, entries: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        enriched: List[Dict[str, Any]] = []

        for e in entries: 
            authors = e.get("author") or []
            copy_e = dict(e)

            copy_e['authors_id'] = self.preprocess_authorid(authors)
            copy_e['authors_info'] = self.preprocess_author(authors)
            copy_e['authors_info_with_id'] = self.preprocess_author_with_id(authors)
            enriched.append(copy_e)

        return enriched

    def to_dataframe(self) -> pd.DataFrame:
        """
        Normalise 'entries' to DataFrame, apply enrichments and renames.
        Returns empty DataFrame (with expected columns if there are fields_map) when there are no results.
        """

        entries = len(self._data)
        if not entries: 
            logger.info("Elsevier response with no results: 0 entries.")
            if fields:
                return pd.DataFrame(columns=list(fields.values()))
            return pd.DataFrame()
        
        logger.info("Normalising %d entries from Elsevier.", entries)
        enriched = self._enrich_entries(self.data)
        df = pd.json_normalize(enriched)

        if fields:
            original_cols = set(df.columns)
            wanted = set(fields.keys())
            not_mapped = original_cols - wanted
            missing_in_df = wanted - original_cols

            if not_mapped:
                logger.info(
                    "Columns present and not mapped (to be ignored): %s",
                    ", ".join(sorted(map(str, not_mapped)))
                )
            if missing_in_df:
                logger.info(
                    "Expected columns in fields_map that did NOT arrive: %s",
                    ", ".join(sorted(map(str, missing_in_df))) 
                )

        keep_cols = [c for c in df.columns if c in fields]
        df = df[keep_cols].rename(columns=fields)

        return df

    def execute(self) -> pd.DataFrame:
        """
        Run the entire pipeline assuming local file input:
        load() → validate contract → enrich → normalise → rename/filter → DataFrame.
        """
        if self._data is None:
            self.load()
        return self.to_dataframe()

In [8]:
prueba = Elsjson(PATH)

# prueba._enrich_entries(prueba.entries)
prueba.execute().head(10)

,Title,Volume,Date,DOI,Cited by,Document Type,Author(s) ID,Authors,Author full names,Issue
0,A Low Power Non-invasive Wrist-Based Approach ...,2392 CCIS,2026-01-01,10.1007/978-3-031-98287-3_10,0,Conference Paper,57219781461;57117284600;36142156300;60102617300,Tobar-Subia-Contento L.M.;Vargas R.;Romero L.A...,Tobar-Subia-Contento L.M. (57219781461);Vargas...,NaN
1,Biogeochemical study of the periglacial slopes...,443,2026-01-01,10.1016/j.icarus.2025.116783,0,Article,58310191500;58310000000;7003652519;60054445900...,Leal M.A.;Tovar D.;de Pablo M.A.;Bonilla M.A.;...,Leal M.A. (58310191500);Tovar D. (58310000000)...,NaN
2,Incorporation of the GRG-optimization method i...,708,2025-12-25,10.1016/j.apcata.2025.120562,0,Article,57024211000;57753440800;23484999200;3548811750...,Castilla-Caballero D.;Martínez-Castro V.;Colin...,Castilla-Caballero D. (57024211000);Martínez-C...,NaN
3,Enhancing consistency in piping and instrument...,7,2025-12-01,10.1016/j.sasc.2025.200373,0,Article,60036445200;60036445300;60035839800;6003598710...,Gómez-Vega F.S.;Acuña O.;Camargo A.C.;Jimenez ...,Gómez-Vega F.S. (60036445200);Acuña O. (600364...,NaN
4,“The professionals weren’t from here”: provisi...,83,2025-12-01,10.1186/s13690-025-01561-z,0,Article,56491243500;59360499800;59361500800;5936083380...,Rubio-León D.C.;Cano-Sierra L.;Reyes-Rivera M....,Rubio-León D.C. (56491243500);Cano-Sierra L. (...,1
5,Microbiological analysis of cigarette butts an...,15,2025-12-01,10.1038/s41598-025-91488-w,0,Article,55258973100;8693393400;55851674100;56674579200...,Díaz-Mendoza C.;Mouthon-Bello J.;Botero C.M.;A...,Díaz-Mendoza C. (55258973100);Mouthon-Bello J....,1
6,Fast computation of 3D domain integrals in bou...,179,2025-10-01,10.1016/j.enganabound.2025.106430,0,Article,57224404584;24537991200,Narváez A.;Useche J.,Narváez A. (57224404584);Useche J. (24537991200),NaN
7,Combined Effect of ABL Profile and Rotation in...,18,2025-09-01,10.3390/en18174726,0,Article,57218161727;58401819600;60092820800;2453799120...,Martinez-Trespalacios J.A.;Barile D.A.;Millan-...,Martinez-Trespalacios J.A. (57218161727);Baril...,17
8,Calibration of multimodal 3D structured-light ...,64,2025-09-01,10.1364/AO.569536,0,Article,57195681584;57117284600;57221475394;3578459680...,Benjumea E.;Vargas R.;Quintero F.;Juarez-Salaz...,Benjumea E. (57195681584);Vargas R. (571172846...,25
9,Optimizing treatment to control LDL cholestero...,195,2025-09-01,10.1016/j.compbiomed.2025.110599,0,Article,59952122700;57191333650;59952159100;57488328300,Yepez D.B.;Porta D.S.;Aguas L.M.;Jattin F.M.,Yepez D.B. (59952122700);Porta D.S. (571913336...,NaN
